# Phase 6: Final Figure Pack (OSP, Exact OSP, Learnable)

Goal: create publication-ready qualitative figures for the three final Phase 11 methods only.


# Learnable MSFA Research Track

This notebook is part of the 10-day publication-oriented extension track.

## Run discipline
- Keep one experiment purpose per notebook.
- Save metrics and checkpoints to the configured project root.
- Do not silently change data splits or loss definitions across notebooks.
- When a result becomes final, export both numeric artifacts and a figure/table asset.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path("/content/drive/MyDrive/Msa-Osp")
FIG_DIR = PROJECT_ROOT / "phase6_figures"

CKPT_OSP = PROJECT_ROOT / "phase11_osp_best.pth"
CKPT_EXACT = PROJECT_ROOT / "phase11_exact_selector_best.pth"
CKPT_LEARN = PROJECT_ROOT / "phase11_learnable_best.pth"
TILE_SOFT_LEARN = PROJECT_ROOT / "phase11_learnable_tile_soft.npy"

BAND_COUNT = 16
WAVELENGTHS = np.linspace(400.0, 700.0, BAND_COUNT, dtype=np.float32)

for p in [CKPT_OSP, CKPT_EXACT, CKPT_LEARN]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required checkpoint: {p}")

FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Figure directory:", FIG_DIR)

In [ ]:
def load_hard_tile_from_ckpt(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if "hard_tile" not in ckpt:
        raise KeyError(f"Checkpoint has no 'hard_tile': {path}")
    tile = np.array(ckpt["hard_tile"], dtype=np.int32)
    if tile.min() == 0:
        tile = tile + 1
    return ckpt, tile

def hard_to_soft(tile_one_based):
    return np.eye(BAND_COUNT, dtype=np.float32)[tile_one_based - 1].transpose(2, 0, 1)

def centroid_map_nm(soft_tile):
    return (soft_tile * WAVELENGTHS[:, None, None]).sum(axis=0).astype(np.float32)

def band_usage(tile_one_based):
    counts = np.bincount(tile_one_based.ravel(), minlength=BAND_COUNT + 1)[1:]
    return counts.astype(np.float32) / max(float(counts.sum()), 1.0)

osp_ckpt, osp_tile = load_hard_tile_from_ckpt(CKPT_OSP)
exact_ckpt, exact_tile = load_hard_tile_from_ckpt(CKPT_EXACT)
learn_ckpt, learn_tile = load_hard_tile_from_ckpt(CKPT_LEARN)

osp_soft = hard_to_soft(osp_tile)
exact_soft = hard_to_soft(exact_tile)
if TILE_SOFT_LEARN.exists():
    learn_soft = np.load(TILE_SOFT_LEARN).astype(np.float32)
    if learn_soft.shape[0] != BAND_COUNT:
        raise ValueError(f"Unexpected learnable soft tile shape: {learn_soft.shape}")
else:
    learn_soft = hard_to_soft(learn_tile)

osp_centroid = centroid_map_nm(osp_soft)
exact_centroid = centroid_map_nm(exact_soft)
learn_centroid = centroid_map_nm(learn_soft)

usage_osp = band_usage(osp_tile)
usage_exact = band_usage(exact_tile)
usage_learn = band_usage(learn_tile)

changed_vs_osp = float((learn_tile != osp_tile).mean())
changed_vs_exact = float((learn_tile != exact_tile).mean())
print(f"Learnable changed fraction vs OSP:   {changed_vs_osp:.4f}")
print(f"Learnable changed fraction vs Exact: {changed_vs_exact:.4f}")

In [ ]:
fig1 = plt.figure(figsize=(15, 4))
for i, (title, tile) in enumerate([("OSP", osp_tile), ("Exact OSP", exact_tile), ("Learnable", learn_tile)], start=1):
    ax = plt.subplot(1, 3, i)
    im = ax.imshow(tile, cmap="tab20", vmin=1, vmax=BAND_COUNT)
    ax.set_title(f"{title} hard tile")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig1.tight_layout()
fig1.savefig(FIG_DIR / "phase6_hard_tiles_phase11.png", dpi=220, bbox_inches="tight")
plt.show()

centroids = [osp_centroid, exact_centroid, learn_centroid]
vmin = min(float(c.min()) for c in centroids)
vmax = max(float(c.max()) for c in centroids)

fig2 = plt.figure(figsize=(15, 4))
for i, (title, centroid) in enumerate([("OSP", osp_centroid), ("Exact OSP", exact_centroid), ("Learnable", learn_centroid)], start=1):
    ax = plt.subplot(1, 3, i)
    im = ax.imshow(centroid, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"{title} centroid wavelength")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="nm")
fig2.tight_layout()
fig2.savefig(FIG_DIR / "phase6_centroid_maps_phase11.png", dpi=220, bbox_inches="tight")
plt.show()

learn_vs_osp = (learn_tile != osp_tile).astype(np.float32)
learn_vs_exact = (learn_tile != exact_tile).astype(np.float32)
x = np.arange(1, BAND_COUNT + 1)

fig3 = plt.figure(figsize=(16, 4))
ax1 = plt.subplot(1, 3, 1)
im1 = ax1.imshow(learn_vs_osp, cmap="gray", vmin=0.0, vmax=1.0)
ax1.set_title("Learnable vs OSP (changed=1)")
ax1.set_xticks([])
ax1.set_yticks([])
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

ax2 = plt.subplot(1, 3, 2)
im2 = ax2.imshow(learn_vs_exact, cmap="gray", vmin=0.0, vmax=1.0)
ax2.set_title("Learnable vs Exact OSP (changed=1)")
ax2.set_xticks([])
ax2.set_yticks([])
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

ax3 = plt.subplot(1, 3, 3)
ax3.plot(x, usage_osp, marker="o", label="OSP")
ax3.plot(x, usage_exact, marker="o", label="Exact OSP")
ax3.plot(x, usage_learn, marker="o", label="Learnable")
ax3.set_xlabel("Band index")
ax3.set_ylabel("Usage fraction")
ax3.set_title("Band usage distribution")
ax3.grid(alpha=0.25)
ax3.legend()

fig3.tight_layout()
fig3.savefig(FIG_DIR / "phase6_differences_and_usage_phase11.png", dpi=220, bbox_inches="tight")
plt.show()

print("Saved figures:")
print(" -", FIG_DIR / "phase6_hard_tiles_phase11.png")
print(" -", FIG_DIR / "phase6_centroid_maps_phase11.png")
print(" -", FIG_DIR / "phase6_differences_and_usage_phase11.png")

These figures are the final qualitative pack for the three Phase 11 methods: OSP, Exact OSP, and Learnable.

They are exported separately and do not overwrite original MATLAB OSP paper figures.
